In [1]:
import sqlite3
import pandas as pd

import numpy as np
try:
    from scipy import stats
except ModuleNotFoundError:
    import warnings
    stats = None
    warnings.warn(
        "scipy is not installed in this environment. Install it with 'pip install scipy' to use statistical tests.",
        RuntimeWarning,
    )

con = sqlite3.connect('../data/raw/airtel.db') 
def q(sql):
    return pd.read_sql(sql, con)

In [2]:
def pval(z, tail='two'):                       # p from a z value
    if tail == 'greater':                      # H1: bigger
        return stats.norm.sf(z)                # area to the right
    if tail == 'less':                         # H1: smaller
        return stats.norm.cdf(z)               # area to the left
    return 2 * stats.norm.sf(abs(z))           # both sides
 
def f_test(x, y):                              # two variances
    f = np.var(x, ddof=1) / np.var(y, ddof=1)  # ratio of spreads
    d1, d2 = len(x) - 1, len(y) - 1            # degrees of freedom
    lo = stats.f.cdf(f, d1, d2)                # area to the left
    p = 2 * min(lo, stats.f.sf(f, d1, d2))     # two-tailed
    print(f'F = {f:.4f}  p = {p:.4g}')         # the result
 
def z_mean2(x, y, tail='two'):                 # two means, large
    se = np.sqrt(np.var(x, ddof=1) / len(x)    # standard error,
                 + np.var(y, ddof=1) / len(y)) # each group its own
    z = (np.mean(x) - np.mean(y)) / se         # gap, in se units
    print(f'z = {z:.4f}  p = {pval(z, tail):.4g}')  # the result

def z_prop2(k1, n1, k2, n2, tail='two'):       # two percentages
    pool = (k1 + k2) / (n1 + n2)               # both groups as one
    se = np.sqrt(pool * (1 - pool) * (1/n1 + 1/n2))  # spread if H0
    z = (k1/n1 - k2/n2) / se                   # gap, in se units
    print(f'z = {z:.4f}  p = {pval(z, tail):.4g}')  # the result

In [3]:
#Claim :the two groups used the same amount of data before the offer 
#H₀ :both groups have the same true average daily data before May
#H₁ :the two averages differ
#Tail :two-tailed — a gap either way would break the split
q(""" SELECT c.ab_group AS g, u.customer_id,
     AVG(u.data_mb) AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id
WHERE u.usage_date < '2025-05-01'
GROUP BY u.customer_id;""")

,g,customer_id,m
0,B,1,729.705882
1,A,2,256.590909
2,A,3,1914.133333
3,B,4,218.880000
4,B,5,980.926667
...,...,...,...
21988,A,21996,6681.156863
21989,A,21997,1460.978571
21990,B,21998,69.021429
21991,A,21999,1153.977778


In [4]:
pre=q(""" SELECT c.ab_group AS g, u.customer_id,
     AVG(u.data_mb) AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id
WHERE u.usage_date < '2025-05-01'
GROUP BY u.customer_id;""")
pre

,g,customer_id,m
0,B,1,729.705882
1,A,2,256.590909
2,A,3,1914.133333
3,B,4,218.880000
4,B,5,980.926667
...,...,...,...
21988,A,21996,6681.156863
21989,A,21997,1460.978571
21990,B,21998,69.021429
21991,A,21999,1153.977778


In [5]:
import sys
import subprocess
if stats is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scipy"])
    from scipy import stats

a=pre[pre.g=='A'].m
b=pre[pre.g=='B'].m

if stats is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scipy"])

f_test(a, b)

F = 1.1108  p = 3.673e-08


In [6]:
a=pre[pre.g=='A'].m
b=pre[pre.g=='B'].m

f_test(a, b)
z_mean2(a, b,)

F = 1.1108  p = 3.673e-08
z = 0.5191  p = 0.6037


In [7]:
cf=q("""
SELECT * FROM customer_features;

""")
A,B=cf[cf.ab_group== 'A'],cf[cf.ab_group== 'B']

for col in ['stated_arpu','tenure_days','complaint_count']:
    print(col)

stated_arpu
tenure_days
complaint_count


In [8]:
cf=q("""
SELECT * FROM customer_features;

""")
A,B=cf[cf.ab_group== 'A'],cf[cf.ab_group== 'B']

for col in ['stated_arpu','tenure_days','complaint_count']:
    print(col)
    f_test(A[col],B[col])

stated_arpu
F = 1.0668  p = 0.0006932
tenure_days
F = 1.0244  p = 0.2061
complaint_count
F = 1.0240  p = 0.2128


In [9]:
rev = q("""
SELECT
    c.ab_group AS g,
    COALESCE(SUM(r.amount_inr), 0) AS s
FROM customer_features c
LEFT JOIN recharges r
    ON r.customer_id = c.customer_id
    AND r.recharge_date >= '2025-01-01'
    AND r.recharge_date < '2025-05-01'
GROUP BY c.customer_id;
""")
rev

,g,s
0,B,897
1,A,198
2,A,3408
3,B,0
4,B,1438
...,...,...
21995,A,6891
21996,A,719
21997,B,836
21998,A,798


In [10]:
A=rev[rev.g=='A'].s
B=rev[rev.g=='B'].s

f_test(B,A)
z_mean2(B,A)

F = 0.9513  p = 0.008866
z = -1.3490  p = 0.1773


In [11]:
q("""SELECT ab_group, COUNT(*) AS n,
            SUM(device_type = 'Smartphone') AS smart,
            SUM(connection_type = 'Prepaid') AS prepaid
     FROM   customer_features GROUP BY ab_group; """)

,ab_group,n,smart,prepaid
0,A,10996,9417,8575
1,B,11004,9554,8574


In [12]:
#smartphone 
z_prop2(9554,11004,9417,10996)
#prepaid
z_prop2(8574,11004,8575,10996)

z = 2.5457  p = 0.01091
z = -0.1177  p = 0.9063


In [13]:
#Claim : the offer raised daily data usage
#H₀ :both groups have the same true average daily data in May and June
#H₁ :group B's average is higher
#Tail :one-tailed — the company said so in advance
off=q("""
SELECT c.ab_group AS g, u.customer_id,
       AVG(u.data_mb) AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id
WHERE u.usage_date >= '2025-05-01'
GROUP BY u.customer_id;
""")
off

,g,customer_id,m
0,A,3,1758.756250
1,B,5,1699.688889
2,A,6,339.763333
3,A,7,5363.175000
4,B,8,1376.940000
...,...,...,...
18443,B,21993,1936.235714
18444,A,21996,6483.050000
18445,A,21997,1262.429032
18446,B,21998,86.217391


In [14]:
A=off[off.g=='A'].m
B=off[off.g=='B'].m 
A,B






(0        1758.756250
 2         339.763333
 3        5363.175000
 9        3828.633333
 10        580.526316
             ...     
 18439    1132.084375
 18440     900.400000
 18444    6483.050000
 18445    1262.429032
 18447    1588.607500
 Name: m, Length: 9247, dtype: float64,
 1        1699.688889
 4        1376.940000
 5        1865.279310
 6        1216.876316
 7        1041.106667
             ...     
 18436    1049.761905
 18441    1922.758621
 18442    1033.139130
 18443    1936.235714
 18446      86.217391
 Name: m, Length: 9201, dtype: float64)

In [15]:
A=off[off.g=='A'].m
B=off[off.g=='B'].m 

f_test(A,B)
z_mean2(A,B)
#HENCE REJECT H0

F = 0.6197  p = 8.402e-116
z = -19.4496  p = 2.936e-84


In [16]:
#Claim : the offer reduced churn
#H₀ :both groups have the same true churn rate
#H₁ :group B's churn rate is lower
#Tail : one-tailed — the company said so in advance
q("""SELECT ab_group, COUNT(*) AS n,
            SUM(is_churned) AS all_churn,
            SUM(churn_date >= '2025-05-01') As may_on
     FROM   customer_features GROUP BY ab_group; """)

,ab_group,n,all_churn,may_on
0,A,10996,2587,905
1,B,11004,2639,914


In [17]:
z_prop2(2639,11004,2587,10996)
#from may churn


z = 0.7937  p = 0.4274


In [18]:
#Claim :the offer raised revenue
#H₀ :both groups paid the same on average
#H₁ :group B paid more
#Tail : one-tailed — the company said so in advance
rev=q("""SELECT c.ab_group AS g,
     COALESCE(SUM(r.amount_inr), 0) AS s
     FROM customer_features c
LEFT JOIN recharges r
     ON r.customer_id = c.customer_id
     AND r.recharge_date >= '2025-05-01'
     GROUP BY c.customer_id;""")
rev

,g,s
0,B,0
1,A,0
2,A,1665
3,B,0
4,B,0
...,...,...
21995,A,3587
21996,A,0
21997,B,209
21998,A,0


In [19]:
a=rev[rev.g=='A'].s
b=rev[rev.g=='B'].s
a,b

(1           0
 2        1665
 5         396
 6        3576
 13       3529
          ... 
 21989       0
 21995    3587
 21996       0
 21998       0
 21999       0
 Name: s, Length: 10996, dtype: int64,
 0          0
 3          0
 4          0
 7          0
 8        349
         ... 
 21991    992
 21992    349
 21993      0
 21994      0
 21997    209
 Name: s, Length: 11004, dtype: int64)

In [20]:
f_test(b,a)
z_mean2(b,a)

F = 0.9697  p = 0.1068
z = -0.5927  p = 0.5534


In [21]:
stats.ttest_ind(b,a,alternative='greater')

TtestResult(statistic=np.float64(-0.5926646653701372), pvalue=np.float64(0.7232941573761509), df=np.float64(21998.0))

In [22]:
#Claim : prepaid and postpaid subscribers churn at different rates
#H₀ :both have the same true churn rate
#H₁ :the two rates differ
q("""SELECT connection_type AS grp, COUNT(*) AS n,
        SUM(is_churned) AS k
        FROM customer_features
        GROUP BY connection_type;""")

,grp,n,k
0,Postpaid,4851,785
1,Prepaid,17149,4441


In [23]:
z_prop2(4441,17149, 785,4851)

z = 14.0364  p = 9.333e-45


In [24]:
#Claim :subscribers with an open ticket churn at a different rate from those who never complained
##H₀ :both have the same true churn rate
##H₁ :the two rates differ
q("""
SELECT CASE WHEN  COMPLAINT_COUNT = 0 THEN 'never'
            WHEN unresolved_count = 0 THEN 'all closed'
            ELSE 'open' END AS grp,
      COUNT(*) AS n,SUM(is_churned) AS k
        FROM customer_features GROUP BY grp;""")   


,grp,n,k
0,all closed,8689,2064
1,never,6631,1328
2,open,6680,1834


In [25]:
z_prop2(1834,6680,1328,6631)

z = 10.0684  p = 7.623e-24


In [26]:
#Claim :year-0 and year-3 subscribers churn at different rates
#H₀ :they share one true churn rate
#H₁ :they do not
q("""SELECT tenure_years AS grp, COUNT(*) AS n,
      SUM(is_churned) AS k
        FROM customer_features
        GROUP BY tenure_years ;""")



,grp,n,k
0,0,4610,1328
1,1,6961,1725
2,2,7055,1541
3,3,3374,632


In [27]:
z_prop2(1328,4610,632,3374)
# p-value < 0.05, reject H0, the two rates differ   

z = 10.3330  p = 4.995e-25


In [28]:
#laim :Himachal Pradesh and Punjab churn at different rates
#H₀ :both have the same true churn rate
#H₁ :the two rates differ
q("""SELECT circle_name AS grp,count(*) AS n,
        SUM(is_churned) AS k
            FROM customer_features
            WHERE circle_name IN ('Himachal Pradesh','Punjab')
            GROUP BY circle_name;""")

,grp,n,k
0,Himachal Pradesh,1247,417
1,Punjab,558,77


In [29]:
z_prop2(408,1412,356,1284)

z = 0.6729  p = 0.501


In [30]:
#Claim :smartphone and feature-phone users churn at different rates
#H₀ :both have the same true churn rate
#H₁ :the two rates differ
q("""SELECT device_type AS grp,count(*) AS n,
        SUM(is_churned) AS k
            FROM customer_features
            WHERE device_type IN ('Smartphone','Feature Phone')
            GROUP BY device_type;""")

,grp,n,k
0,Feature Phone,3029,719
1,Smartphone,18971,4507


In [31]:
z_prop2(4507,18971,719,3029)

z = 0.0241  p = 0.9807


In [32]:
# Claim :gender and device type each change the data a subscriber uses
# H₀ :both groups have the same true average daily data
# H₁ :the two averages differ

gen=q("""
SELECT c.gender,c.device_type,
        u.data_mb AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id;
""")
dev=q("""SELECT c.gender,c.device_type,
        u.data_mb AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id
GROUP BY c.customer_id;

""")
dev

,gender,device_type,m
0,Male,Smartphone,571.8
1,Male,Smartphone,190.0
2,Female,Smartphone,726.2
3,Male,Feature Phone,190.6
4,Female,Smartphone,928.0
...,...,...,...
21991,Male,Smartphone,5671.3
21992,Male,Smartphone,2264.7
21993,Female,Feature Phone,63.6
21994,Female,Smartphone,1265.3


In [33]:
male=gen[gen.gender=='Male'].m
female=gen[gen.gender=='Female'].m

f_test(female,male)
#p value <0.05
#we can go for z test
z_mean2(female,male)
# p value <0.05 so significant

F = 0.9315  p = 6.262e-177
z = -7.2519  p = 4.109e-13


In [34]:
fd=dev[dev.device_type=='Feature Phone'].m
sp=dev[dev.device_type=='Smartphone'].m

f_test(fd,sp)
z_mean2(fd,sp)


F = 0.0265  p = 0
z = -94.3706  p = 0


In [35]:
gen = q("""
    SELECT c.gender, c.device_type,
        u.data_mb AS m
    FROM usage_daily u
    JOIN customer_features c ON c.customer_id = u.customer_id;
""")
gen

dev = q("""
    SELECT c.gender, c.device_type,
        AVG(u.data_mb) AS m
    FROM usage_daily u
    JOIN customer_features c ON c.customer_id = u.customer_id
    GROUP BY c.customer_id;
""")
dev

,gender,device_type,m
0,Male,Smartphone,729.705882
1,Male,Smartphone,256.590909
2,Female,Smartphone,1868.932727
3,Male,Feature Phone,218.880000
4,Female,Smartphone,1146.794872
...,...,...,...
21991,Male,Smartphone,6610.941772
21992,Male,Smartphone,1400.037624
21993,Female,Feature Phone,75.106154
21994,Female,Smartphone,1153.977778


In [36]:
fd=dev[dev.device_type=='Feature Phone'].m
sp=dev[dev.device_type=='Smartphone'].m

f_test(fd,sp)

z_mean2(fd, sp)

F = 0.0268  p = 0
z = -119.2400  p = 0


In [37]:
q("""SELECT
    CASE
        WHEN complaint_count = 0 THEN 'No complaints'
        WHEN unresolved_count > 0 THEN 'Open complaint'
        ELSE 'All resolved'
    END AS grp,
    COUNT(*) AS n,
    SUM(is_churned) AS k
FROM customer_features
WHERE complaint_count = 0 OR unresolved_count > 0
GROUP BY grp;
""")

,grp,n,k
0,No complaints,6631,1328
1,Open complaint,6680,1834


In [38]:
z_prop2(1328,6631,1834,6680)

z = -10.0684  p = 7.623e-24


In [39]:
q("""SELECT
    plan_name AS grp,
    COUNT(*) AS n,
    SUM(is_churned) AS k
FROM customer_features
WHERE plan_name IN (
    'Prepaid Lite 99',
    'Prepaid Smart 155'
)
GROUP BY plan_name;
""")

,grp,n,k
0,Prepaid Lite 99,996,310
1,Prepaid Smart 155,1137,343


In [40]:
z_prop2(310,996,343,1137)

z = 0.4786  p = 0.6322


In [41]:
q("""SELECT
    circle_name AS grp,
    COUNT(*) AS n,
    SUM(is_churned) AS k
FROM customer_features
WHERE circle_name IN (
    'Himachal Pradesh',
    'Maharashtra'
)
GROUP BY circle_name;
""")

,grp,n,k
0,Himachal Pradesh,1247,417
1,Maharashtra,1428,470


In [42]:
z_prop2(417,1247,470,1428)

z = 0.2889  p = 0.7727
